In [ ]:
# Google Drive をマウント（チェックポイントの退避先。セッション切れ対策）
from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} / VRAM {p.total_memory/1e9:.1f} GB')
else:
    print('GPU なし（ランタイムのタイプを GPU に変更してください）')


In [1]:
!rm -rf suzume-dsa/
!git clone https://github.com/off4321/suzume-dsa.git

Cloning into 'suzume-dsa'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 133 (delta 51), reused 133 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 233.05 KiB | 23.30 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [2]:
# --- torch を GPU 有無で自動選択 ---------------------------------
# GPU があれば CUDA 版 torch (cu124 グループ)、無ければ CPU 版(デフォルト)を
# uv に選ばせる。以降の `uv run` セルでは `uv run $UV_TORCH ...` の形で参照する。
import os
import subprocess


def _detect_gpu() -> bool:
    if os.path.exists('/dev/nvidia0'):
        return True
    if os.environ.get('COLAB_GPU'):
        return True
    try:
        return subprocess.run('nvidia-smi', shell=True, capture_output=True,
                              timeout=10).returncode == 0
    except Exception:
        return False


_has_gpu = _detect_gpu()
os.environ['UV_TORCH'] = '--no-group cpu --group cu124' if _has_gpu else ''
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # 断片化OOM対策

# gated データセットを使う場合は HF_TOKEN も環境変数へ（未登録ならスキップ）
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN: 設定済み')
except Exception:
    print('HF_TOKEN: 未設定（公開データセットのみ使用可）')
print('torch:', 'cu124 (GPU)' if _has_gpu else 'cpu (default)')


HF_TOKEN: 未設定（公開データセットのみ使用可）
torch: cu124 (GPU)


In [3]:
# 検証: uv 環境の torch が CUDA 版かを本番前に確認する
import os
get_ipython().system(
    'cd suzume-dsa/src && uv run ' + os.environ.get('UV_TORCH', '') +
    ' python -c "import torch; print(torch.__version__, torch.cuda.is_available())"')


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: /content/suzume-dsa/.venv
Installed 59 packages in 195ms                                       
2.6.0+cu124 True


In [ ]:
# ============================================================
# パラメータ管理（定数集約）
#   モデル規模は PRESET の 1 行だけで切替。アーキ寸法そのものは
#   src/suzume_dsa/config.py の PRESETS（4b / 05b / tiny）が真実の値で、
#   count_params / vram_calibrate / suzume-dsa-train はすべて --preset "$PRESET" で受ける。
#   ここが持つのは「どのプリセットか」＋規模ごとの“学習の回し方”だけ（寸法は重複させない）。
#   SFT / export は checkpoint の cfg を継承するので、寸法指定は事前学習の 1 回だけでよい。
#   投入データセットは次セル（PRETRAIN_DATASETS / SFT_DATASETS）で一元定義。
# ============================================================
PRESET = "05b"          # "05b"(total460M/active201M) / "4b"(total4.1B/active1.5B) / "tiny"(疎通用)

VOCAB_SIZE = 32000      # トークナイザ語彙。規模非依存（4B/0.5B で同じ 1 本を流用）
# トークナイザは Drive に置く＝セッション切れ後の --resume でも同じ語彙を使い回せる（重要）
TOKENIZER_PREFIX = "/content/drive/MyDrive/suzume_dsa/tokenizer/sp"   # → <prefix>.model
TOKENIZER_MAX_SAMPLES = 200_000   # SP 学習の各データセット上限（語彙学習は全量不要）

# 学習効率: MTP（Multi-Token Prediction）補助損失。config.py のプリセットは既定 OFF
# (mtp_depth=0) なので、ここで有効化する。train-only で export では捨てられる＝GGUF 不変。
# 事前学習で mtp_depth>0 にすると、その MTP は checkpoint 経由で SFT にも継承され学習される。
MTP_DEPTH = 1   # 0 で無効。1〜2 で次トークン+先読みの補助損失（サンプル効率↑）

# --- 規模ごとの学習ハイパラ（README の 0.5B レシピ準拠。PRESET で自動選択） ---
#   MAX_SAMPLE は「データセット1本あたり」の読込上限（全 spec に個別適用）。
#   block を伸ばす step で batch を下げ、系列長×バッチのカリキュラムで VRAM をほぼ一定化。
#   SFT_* も事前学習と同じ効率化（Muon/WSD/MTP継承/選択backprop）を共有する。
_HP = {
    "05b": dict(
        MAX_SAMPLE=1_000_000, STEPS=20000,
        BLOCK_SIZE_SCHEDULE="0%:1024,50%:2048,80%:4096",
        BATCH_SIZE_SCHEDULE="0%:48,50%:24,80%:12",
        OPTIMIZER="muon", MUON_LR=0.02, LEARNING_RATE=2e-4,
        LR_SCHEDULE="wsd", WARMUP_STEPS=400, SELECT_TOPP=1.0,
        SFT_STEPS=2000, SFT_BATCH_SIZE=8, SFT_MAX_LEN=2048,
        SFT_LEARNING_RATE=1e-5, SFT_CKPT_EVERY=500,
        SFT_OPTIMIZER="muon", SFT_MUON_LR=0.01, SFT_LR_SCHEDULE="wsd",
        SFT_WARMUP=100, SFT_SELECT_TOPP=1.0,
    ),
    "4b": dict(
        MAX_SAMPLE=700_000, STEPS=50000,
        BLOCK_SIZE_SCHEDULE="0%:1024,50%:2048,80%:4096",
        BATCH_SIZE_SCHEDULE="0%:20,50%:10,80%:5",
        OPTIMIZER="muon", MUON_LR=0.005, LEARNING_RATE=1.5e-4,
        LR_SCHEDULE="wsd", WARMUP_STEPS=400, SELECT_TOPP=1.0,
        SFT_STEPS=6000, SFT_BATCH_SIZE=8, SFT_MAX_LEN=2048,
        SFT_LEARNING_RATE=2e-5, SFT_CKPT_EVERY=500,
        SFT_OPTIMIZER="muon", SFT_MUON_LR=0.005, SFT_LR_SCHEDULE="wsd",
        SFT_WARMUP=100, SFT_SELECT_TOPP=1.0,
    ),
    "tiny": dict(
        MAX_SAMPLE=2000, STEPS=200,
        BLOCK_SIZE_SCHEDULE="0%:256", BATCH_SIZE_SCHEDULE="0%:8",
        OPTIMIZER="adamw", MUON_LR=0.02, LEARNING_RATE=3e-4,
        LR_SCHEDULE="cosine", WARMUP_STEPS=0, SELECT_TOPP=1.0,
        SFT_STEPS=200, SFT_BATCH_SIZE=4, SFT_MAX_LEN=512,
        SFT_LEARNING_RATE=1e-5, SFT_CKPT_EVERY=100,
        SFT_OPTIMIZER="adamw", SFT_MUON_LR=0.02, SFT_LR_SCHEDULE="cosine",
        SFT_WARMUP=10, SFT_SELECT_TOPP=1.0,
    ),
}
globals().update(_HP[PRESET])   # 選んだ規模のハイパラを大文字定数として展開
# ※ suzume-dsa-train の checkpoint 間隔は CLI 非公開（train.py 内で固定 200 step）。
# ※ SELECT_TOPP / SFT_SELECT_TOPP は 1.0=無効。<1.0 で損失上位トークンだけ backprop。
#    SFT は教師トークンが元々疎なので 1.0 のまま推奨（下げるなら 0.7 程度から）。

# --- 効率化（事前学習・SFT 共通。bf16 は CUDA 時のみ効く＝CPU は自動 fp32） ---
PRECISION = "bf16"       # "bf16"(推奨,約2倍速/半メモリ) / "fp16" / "fp32"
GRAD_ACCUM = 1           # 勾配累積。実効 batch = batch×accum（VRAM が厳しければ増やす）
Z_LOSS = 1e-4            # 出力 z-loss（安定化・高LR可）。0 で無効
ROUTER_Z = 1e-3          # MoE ルーター z-loss（ルーティング安定化）。0 で無効
SFT_PACK = True          # SFT シーケンスパッキング（境界マスク付き。パディング無駄を消す）
SFT_NEFTUNE = 5.0        # NEFTune 埋め込みノイズ（指示追従↑）。0 で無効
SFT_Z_LOSS = 1e-4
SFT_ROUTER_Z = 1e-3

# --- 選好最適化 (ORPO) 段。SFT の後に chosen/rejected で磨く（参照モデル不要） ---
ORPO_STEPS = 1000
ORPO_BATCH_SIZE = 4
ORPO_MAX_LEN = 1024
ORPO_LR = 5e-6
ORPO_LAM = 0.1           # 選好項 L_OR の重み
ORPO_OUTPUT_DIR = f"/content/drive/MyDrive/suzume_dsa_orpo/output_{PRESET}"

# --- アイデンティティ（自己認識）と GGUF 名乗り ---
MODEL_NAME = "suzume-dsa"
MODEL_CREATOR = ""       # 作者句（任意）。実在組織を騙らないよう既定は空
DEFAULT_SYSTEM = "あなたはすずめ(suzume-dsa)、日本語に特化した言語モデルです。"
IDENTITY_N = 400         # 自己紹介 Q&A 生成数（SFT に混ぜる）
IDENTITY_JSONL = "identity.jsonl"   # suzume-dsa/ からの相対（学習コマンドは cd suzume-dsa 済み）
GGUF_OUT = f"/content/drive/MyDrive/suzume_dsa/suzume-{PRESET}.gguf"

# 出力先（規模ごとに分離。Drive 退避でセッション切れに耐える → --resume で途中再開）
OUTPUT_DIR     = f"/content/drive/MyDrive/suzume_dsa/output_{PRESET}"
CKPT_DIR       = f"/content/drive/MyDrive/suzume_dsa_ckpt_{PRESET}"
SFT_OUTPUT_DIR = f"/content/drive/MyDrive/suzume_dsa_sft/output_{PRESET}"
SFT_CKPT_DIR   = f"/content/drive/MyDrive/suzume_dsa_sft_ckpt_{PRESET}"

# Vision (VLM SFT) 用の追加設定（この段は本パイプラインの後段。規模非依存の視覚塔）
VISION_IMAGE_SIZE = 224
VISION_PATCH_SIZE = 16
VISION_DIM = 384
VISION_DEPTH = 6
VISION_HEADS = 6
IMAGE_COLUMN = "image"
VISION_LEARNING_RATE = 2e-5
VISION_STEPS = 3000
VISION_OUTPUT_DIR = f"/content/drive/MyDrive/suzume_dsa_vision/output_{PRESET}"
VISION_CKPT_DIR   = f"/content/drive/MyDrive/suzume_dsa_vision_ckpt_{PRESET}"


import os
for k, v in list(globals().items()):
    if k.isupper() and isinstance(v, (str, int, float)):
        os.environ[k] = str(v)
print(f"PRESET={PRESET}  STEPS={STEPS}  MTP_DEPTH={MTP_DEPTH}  precision={PRECISION}  "
      f"pack={SFT_PACK}  neftune={SFT_NEFTUNE}  out={OUTPUT_DIR}")


In [ ]:
# ============================================================
# 投入データセット（定数一元管理）
#   ここに書いた spec を「下見(check_dataset) → トークナイザ → 事前学習 / SFT / ORPO」で
#   そのまま全部使う。spec 書式: "path[:config][:split][:field]"
#   読めない spec（config 必須・カラム不明など）は各段で警告してスキップ＝1本ダメでも継続。
# ============================================================
PRETRAIN_DATASETS = [
    "mlabonne/open-perfectblend",
    "WithinUsAI/claude_mythos_distilled_25k",
    "lordx64/agentic-distill-fable-5-sft",
    "ai4privacy/pii-masking-openpii-1.5m:::source_text",
    "allenai/c4:ja",
    "hotchpotch/fineweb-2-edu-japanese-scores",
    "izumi-lab/llm-japanese-dataset",
    "christopher/rosetta-code::train:code",
    "google/code_x_glue_ct_code_to_text:python:train:code",
    "m-a-p/CodeFeedback-Filtered-Instruction::train:answer",
    "jtatman/python-code-dataset-500k::train:output",
    "oshizo/japanese-wikipedia-paragraphs",
    "kfsky/competition-math-japanese",
    "kunishou/oasst1-chat-44k-ja",
    "NousResearch/hermes-function-calling-v1",
    "llm-jp/llava-instruct-ja",
    "globis-university/aozorabunko-clean::train:text",
    "hpprc/jawiki",
    "llm-jp/oasst2-33k-ja",
    "Aratako/Magpie-Tanuki-8B-97k",
    "HuggingFaceFW/fineweb-2:jpn_Jpan:train:text",
    "range3/cc100-ja::train:text",
    "wikimedia/wikipedia:20231101.ja:train:text",
    # "oscar-corpus/OSCAR-2301:ja:train:text",
    # "if001/oscar_2023_filtered",
    # "bigcode/the-stack-dedup",
]

# SFT の spec 4 番目フィールドは 3 形式に対応（load_sft_conversations）:
#   ・会話列名        例 "::train:conversations"（messages/conversations をそのまま使う）
#   ・列マッピング DSL 例 "::train:instruction=question,reasoning=reasoning,output=answer"
#                       （別カラム→1会話。reasoning は <think>…</think> で包んで assistant へ）
#   ・"format=harmony" 例 ":cfg:split:format=harmony"（harmony 文字列を messages に復元）
SFT_DATASETS = [
    "kunishou/oasst1-chat-44k-ja::train:conversations",
    "mlabonne/open-perfectblend::train:conversations",
    "WithinUsAI/claude_mythos_distilled_25k::train:messages",
    "NousResearch/hermes-function-calling-v1:glaive_func_calling::conversations",
    "DataPilot/Zero_SFT_Ja_v3.5::train:messages",
    "llm-jp/magpie-sft-v1.0::train:conversations",
    "FreedomIntelligence/sharegpt-japanese::train:conversations",
    "nvidia/Nemotron-SFT-Multilingual-v1::code_ja:messages",
    "nvidia/Nemotron-SFT-Multilingual-v1::math_ja:messages",
    "DeL-TaiseiOzaki/Tengentoppa-sft-reasoning-ja::train",
    "ronantakizawa/Medical-o1-Reasoning-SFT-Japanese::train",
    # 列マッピング DSL / harmony（loader 拡張により対応済み）
    "hotchpotch/japanese-qa-reasoning-100k::train:instruction=question,reasoning=reasoning,output=answer",
    "Kendamarron/magpie-math-japanese-instruction-evolved::train:instruction=problem,reasoning=generated_solutions_reasoning,output=answer",
    "llm-jp/llm-jp-4-thinking-sft-data:nemotron_post_v3_math:reasoning_high:format=harmony",
    "lightblue/distilabel-reasoning-R1-Llama-70B::train:instruction=translated_instruction,output=ja_responses",
]

# ORPO 用の選好データ（chosen / rejected 列を持つもの、または prompt+文字列）。
PREF_DATASETS = [
    "llm-jp/hh-rlhf-12k-ja::train",
    "Aratako/Magpie-Preference-24k-Formatted::train",
    # "shisa-ai/shisa-v2-dpo-ja::train",
]

VISION_DATASETS = [
    # "" ここに VLM SFT 用の image+text データセットを追加
]

print(f"PRETRAIN_DATASETS: {len(PRETRAIN_DATASETS)} 本 / SFT_DATASETS: {len(SFT_DATASETS)} 本 / "
      f"PREF_DATASETS: {len(PREF_DATASETS)} 本")


## パイプラインと「0.5B の指定方法」

規模スイッチは上のセルの **`PRESET` 定数 1 行だけ**。各段がどう 0.5B を受けるか:

| 順 | 段階 | 実行 | 0.5B の指定方法 |
|---|---|---|---|
| 1 | パラメータ見積り | `tools/count_params.py` | **`--preset "$PRESET"`** |
| 2 | VRAM 実測 | `tools/vram_calibrate.py` | **`--preset "$PRESET"`**（count_params のパーサを継承） |
| 3 | データ下見 | `tools/check_dataset.py` | **規模非依存**（指定不要） |
| 4 | トークナイザ | `tools/train_tokenizer.py` | **規模非依存**（`--vocab-size` のみ。4B と同じ 1 本を流用） |
| 5 | 事前学習 | `suzume-dsa-train` | **`--preset "$PRESET"`** |
| 6 | SFT | `suzume-dsa-sft` | **指定不要**（`--init-from` の checkpoint から cfg を継承） |

寸法を意識するのは **1・2・5 の 3 箇所だけ**。トークナイザは規模非依存で共通、SFT/export は checkpoint から寸法（0.5B のまま）を継承する。4B に上げるときは `PRESET="4b"` にするだけで 1〜6 が追随する。


In [6]:
# 1. パラメータ数の見積り（--preset で規模選択。meta 構築なのでメモリ不要）
#    total/active と学習メモリ概算が出る。個別レバー(--n-embd 等)で微調整も可。
!cd suzume-dsa && uv run tools/count_params.py --preset "$PRESET"


⠴  (2/2)                                                                        Uninstalled 2 packages in 185ms
Installed 2 packages in 211ms                               
usage: count_params.py [-h] [--vocab-size VOCAB_SIZE] [--n-embd N_EMBD]
                       [--tie-embeddings | --no-tie-embeddings]
                       [--n-layer N_LAYER]
                       [--n-layer-dense-lead N_LAYER_DENSE_LEAD]
                       [--n-head N_HEAD] [--head-dim-nope HEAD_DIM_NOPE]
                       [--head-dim-rope HEAD_DIM_ROPE]
                       [--head-dim-v HEAD_DIM_V] [--q-lora-rank Q_LORA_RANK]
                       [--kv-lora-rank KV_LORA_RANK] [--rope-base ROPE_BASE]
                       [--n-ctx-train N_CTX_TRAIN]
                       [--indexer-n-head INDEXER_N_HEAD]
                       [--indexer-head-size INDEXER_HEAD_SIZE]
                       [--indexer-top-k INDEXER_TOP_K] [--n-ff N_FF]
                       [--n-ff-exp N_FF_EXP] [--n-expert N_EX

In [7]:
# 2. VRAM 実測キャリブレーション（要 CUDA）: 目標 VRAM に収まる最大 batch を逆算。
#    カリキュラム末尾の最大 block=4096（一番きつい所）で測る。ここで得た batch を
#    上の BATCH_SIZE_SCHEDULE 先頭値の目安にする。--preset は count_params と共通。
import os
for i in [1024, 2048, 4096]:                 # 2048打ち切りなら[1024,2048]。4096も見るなら足す
    os.environ["CUR_BLOCK"] = str(i)   # ← ループ変数を環境変数に入れる
    print(f"\n========== block_size = {i} ==========")
    !cd suzume-dsa && uv run $UV_TORCH tools/vram_calibrate.py \
        --preset "$PRESET" --block-size "$CUR_BLOCK" --precision bf16



========== block_size = 1024 ==========
Uninstalled 2 packages in 212ms
Installed 2 packages in 176ms                               
usage: vram_calibrate.py [-h] [--vocab-size VOCAB_SIZE] [--n-embd N_EMBD]
                         [--tie-embeddings | --no-tie-embeddings]
                         [--n-layer N_LAYER]
                         [--n-layer-dense-lead N_LAYER_DENSE_LEAD]
                         [--n-head N_HEAD] [--head-dim-nope HEAD_DIM_NOPE]
                         [--head-dim-rope HEAD_DIM_ROPE]
                         [--head-dim-v HEAD_DIM_V] [--q-lora-rank Q_LORA_RANK]
                         [--kv-lora-rank KV_LORA_RANK] [--rope-base ROPE_BASE]
                         [--n-ctx-train N_CTX_TRAIN]
                         [--indexer-n-head INDEXER_N_HEAD]
                         [--indexer-head-size INDEXER_HEAD_SIZE]
                         [--indexer-top-k INDEXER_TOP_K] [--n-ff N_FF]
                         [--n-ff-exp N_FF_EXP] [--n-expert N_EXPERT]
       

In [ ]:
!cd suzume-dsa/ && uv add hf_transfer
!export HF_XET_HIGH_PERFORMANCE=1
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
# os.environ["HF_HUB_DISABLE_XET"] = "1"

### 3. データ下見（規模非依存）

`check_dataset.py` はデータの中身・カラム・件数を見るだけで、モデル規模には一切依存しない（0.5B でも 4B でも同じ）。下のセルで本番投入するデータ（`HF_DATASET` / `SFT_DATASET`）が読めることを確認する。


In [ ]:
# データの下見: 上のセルで定義した PRETRAIN_DATASETS / SFT_DATASETS が読めるか・
# どのカラムがテキストかを確認する（規模非依存）。ここで通った spec がそのまま学習に乗る。
for spec in PRETRAIN_DATASETS:
    print("=" * 70, spec, sep="\n")
    # IPython の `!` は変数展開({spec})を先頭行でしか解決しないため、単一行で渡す。
    !cd suzume-dsa/src && uv run $UV_TORCH check_dataset.py '{spec}' --n 2 --max-samples 200 --stream

print("=" * 70, "sft", "=" * 70)

for spec in SFT_DATASETS:
    print("=" * 70, spec, sep="\n")
    !cd suzume-dsa/src && uv run $UV_TORCH check_dataset.py '{spec}' --n 2 --max-samples 200 --stream --sft


### 4. トークナイザ学習（規模非依存）

トークナイザは `VOCAB_SIZE`（=32000）だけで決まり、モデル規模には依存しない。**4B と 0.5B で同じ `sp.model` を共有**する。`PRETRAIN_DATASETS` 全部のテキスト（各 `TOKENIZER_MAX_SAMPLES` 行）で語彙を学習し、`$TOKENIZER_PREFIX.model`（= **Drive 上**）に書く。Drive に置くのは、セッション切れ後の `--resume` で**同じ語彙**を使い回すため（語彙が変わると checkpoint が無効になる）。既に学習済みなら流用（このセルはスキップ）。特殊トークン（FIM / chat）は自動で語彙に含まれる。


In [ ]:
# 4. トークナイザ学習（規模非依存。PRETRAIN_DATASETS 全部のテキストで語彙を作る）
#    出力は Drive の $TOKENIZER_PREFIX.model → セッション切れ後の --resume でも同じ語彙。
#    複数 spec を1本のコマンドに渡すため Python 側で組み立てて実行（`!` の {} 展開の罠回避）。
import shlex

specs = " ".join(shlex.quote(s) for s in PRETRAIN_DATASETS)
cmd = (
    f'cd suzume-dsa && uv run {os.environ.get("UV_TORCH", "")} tools/train_tokenizer.py '
    f'--hf-dataset {specs} '
    f'--hf-max-samples {TOKENIZER_MAX_SAMPLES} '
    f'--vocab-size {VOCAB_SIZE} '
    f'--out {shlex.quote(TOKENIZER_PREFIX)}'
)
print(cmd)
get_ipython().system(cmd)


In [ ]:
# 4b. トークナイザ評価（任意）: 日本語での fertility を測り、他トークナイザと勝負。
#     fertility=tokens/char が小さいほど日本語を短く表せる＝実効コンテキスト/コスト有利。
#     比較したい HF トークナイザがあれば --hf-tokenizer に名前を足す（transformers 必要）。
cmd = (
    f'cd suzume-dsa && uv run $UV_TORCH tools/tokenizer_eval.py '
    f'--sp {shlex.quote(TOKENIZER_PREFIX + ".model")} '
    f'--hf-dataset "range3/wikipedia-ja-20230101" --max-samples 2000'
    # 例: 他モデルと比較 → 末尾に  --hf-tokenizer meta-llama/Llama-2-7b-hf Qwen/Qwen2.5-0.5B
)
print(cmd)
get_ipython().system(cmd)


### 4.5 データ量とトークン予算は足りる？

**「学習で見るトークン数」は corpus サイズではなく `STEPS × (batch×block)` で決まる**（カリキュラムは batch×block をほぼ一定に保つ設計）。0.5B の Chinchilla 最適はおよそ **20 tok/param ≈ 9B トークン**。下のセルで、今の `STEPS` が何トークン・何 tok/param に相当するかを見積もる。

- **既定 `STEPS=20000`（≈1B tok, ~2 tok/param）**: これは README が言う「パイプラインを安く一気通しする検証run」。壊れず GGUF まで出せるが、性能的には大幅に undertrained。
- **ちゃんと 0.5B を鍛えるなら** Chinchilla 級に近づけて `STEPS≈180k`（≈9B tok）。その分 corpus 側も足りる必要があり、`MAX_SAMPLE`（1本あたり上限）を上げるか大型 web 系（c4/cc100/fineweb/wikipedia）中心に増やす。同じ文を 4 エポック程度まで繰り返すのは実用上許容。

つまり **corpus の“本数”は十分**（下見が通れば web 系だけで数十億トークン規模）。**足りるかは `STEPS` の設定次第**。


In [ ]:
# 4.5 トークン予算の見積り（学習で見るトークン数 vs Chinchilla最適）。純計算・学習不要。
#     ※ ノートブック本体の kernel には torch を入れない方針なので、総パラメータは
#       count_params.py（uv 環境）が出す値を使う。ここでは既知値を定数で持つ。
PRESET_TOTAL_PARAMS = {"05b": 460_000_000, "4b": 4_030_000_000, "tiny": 1_000_000}


def _sched_at(spec, steps, s):
    """"0%:1024,50%:2048" のスケジュールで step s のときの値。"""
    pairs = []
    for part in spec.split(","):
        k, _, v = part.partition(":")
        k = k.strip()
        step = round(float(k[:-1]) / 100 * steps) if k.endswith("%") else int(k)
        pairs.append((step, int(v)))
    pairs.sort()
    cur = pairs[0][1]
    for st, val in pairs:
        if s >= st:
            cur = val
    return cur


def tokens_seen(block_sched, batch_sched, steps):
    return sum(_sched_at(block_sched, steps, s) * _sched_at(batch_sched, steps, s)
               for s in range(steps))


n_params = PRESET_TOTAL_PARAMS[PRESET]
seen = tokens_seen(BLOCK_SIZE_SCHEDULE, BATCH_SIZE_SCHEDULE, STEPS)
chinchilla = 20 * n_params
print(f"model total params : {n_params/1e9:.2f}B  (PRESET={PRESET})")
print(f"STEPS={STEPS} で見るトークン: {seen/1e9:.2f}B  ({seen/n_params:.1f} tok/param)")
print(f"Chinchilla最適(×20): {chinchilla/1e9:.1f}B  → 到達には STEPS≈{round(STEPS*chinchilla/seen):,}")
print("※ tok/param が 20 に近いほど良い。検証runなら 1〜2 でも可（安く一気通し）。")


### 5. 事前学習（`--preset "$PRESET"` で規模選択）

ここが唯一「寸法を明示する学習」。`--preset 05b` で 0.5B を選び、系列長×バッチのカリキュラムで VRAM をほぼ一定に保つ。checkpoint は Drive の `$OUTPUT_DIR` に出る（`train.py` 内で 200 step ごと固定。切断時は 5b で `--resume`）。

**内蔵の効率化**（この run で有効）: **Muon** optimizer（`--optimizer muon`）、**WSD** 学習率（`--lr-schedule wsd`）、**系列長/バッチ カリキュラム**、**MTP**（Multi-Token Prediction 補助損失, `--mtp-depth 1`／train-only・GGUF には残らない）、aux-free な MoE ロードバランス、非有限勾配ガード。**任意で追加できるレバー**: `--fim 0.3`（FIM 穴埋め）、`--select-topp 0.7`（損失上位トークンだけ backprop）、`--mup`（μP 幅転移で LR を規模間で転移）。Muon で nan が出たら `--muon-lr` を半分にして 5b で再開。


In [ ]:
# 5. 事前学習: --preset "$PRESET"（=0.5B）。PRETRAIN_DATASETS を全部連結して投入。
#    寸法はここだけで指定、以降（SFT/export）は checkpoint から継承される。
#    効率化: bf16 + Muon + WSD + 系列長/バッチ カリキュラム + MTP + z-loss + 選択backprop。
import shlex

specs = " ".join(shlex.quote(s) for s in PRETRAIN_DATASETS)
cmd = (
    f'cd suzume-dsa && uv run {os.environ.get("UV_TORCH", "")} suzume-dsa-train '
    f'--preset {PRESET} --sp-model {shlex.quote(TOKENIZER_PREFIX + ".model")} '
    f'--hf-dataset {specs} --hf-max-samples {MAX_SAMPLE} '
    f'--steps {STEPS} --mtp-depth {MTP_DEPTH} --select-topp {SELECT_TOPP} '
    f'--precision {PRECISION} --grad-accum {GRAD_ACCUM} '
    f'--z-loss {Z_LOSS} --router-z {ROUTER_Z} '
    f'--block-size-schedule {shlex.quote(BLOCK_SIZE_SCHEDULE)} '
    f'--batch-size-schedule {shlex.quote(BATCH_SIZE_SCHEDULE)} '
    f'--optimizer {OPTIMIZER} --muon-lr {MUON_LR} --lr {LEARNING_RATE} '
    f'--lr-schedule {LR_SCHEDULE} --warmup {WARMUP_STEPS} '
    f'--out {shlex.quote(OUTPUT_DIR)} --device cuda'
)
print(cmd)
get_ipython().system(cmd)
#   → $OUTPUT_DIR/model.pt（+ 200step ごと checkpoint_step*.pt）。追加レバー: --fim 0.3
#     （穴埋め）/ --mup（μP幅転移）。途中で切れたら次セル（5b）で最新 checkpoint から --resume。


### 5b. 途中から再開（`--resume`）

できる。事前学習は **200 step ごとに `$OUTPUT_DIR/checkpoint_step*.pt` を Drive に保存**し、`--resume <ckpt>` で**モデル・optimizer・step を復元**して続きから走る（`train.py`: `start = load_checkpoint(...)` → `range(start, steps)`）。学習率スケジュールと系列長/バッチのカリキュラムは step 基準なので、`--steps` を初回と同じ値にすれば復元後もそのまま整合する。

Colab 復帰時は **先頭セル群（Drive マウント → clone → uv 準備 → 定数セル → データ定数セル）を再実行**してから下のセルを回す（`PRESET` などの定数が必要）。トークナイザは Drive 上にあるので再学習不要。下のセルは `$OUTPUT_DIR` の**最新 checkpoint を自動検出**して再開する。Muon で nan が出たら `--muon-lr` を半分にして再 resume。


In [ ]:
# 5b. 最新 checkpoint から再開（切断時だけ実行）。--steps / --mtp-depth は初回と同じ値に。
import glob
import os
import re
import shlex

ckpts = glob.glob(f"{OUTPUT_DIR}/checkpoint_step*.pt")
assert ckpts, f"{OUTPUT_DIR} に checkpoint_step*.pt が無い（まだ 200 step 未満？そのまま 5. を回す）"


def _step(p):
    return int(re.search(r"step(\d+)", p).group(1))


latest = max(ckpts, key=_step)
print(f"resume from step {_step(latest)}: {latest}")

specs = " ".join(shlex.quote(s) for s in PRETRAIN_DATASETS)
cmd = (
    f'cd suzume-dsa && uv run {os.environ.get("UV_TORCH", "")} suzume-dsa-train '
    f'--preset {PRESET} --sp-model {shlex.quote(TOKENIZER_PREFIX + ".model")} '
    f'--hf-dataset {specs} --hf-max-samples {MAX_SAMPLE} '
    f'--steps {STEPS} --mtp-depth {MTP_DEPTH} --select-topp {SELECT_TOPP} '
    f'--precision {PRECISION} --grad-accum {GRAD_ACCUM} '
    f'--z-loss {Z_LOSS} --router-z {ROUTER_Z} '
    f'--block-size-schedule {shlex.quote(BLOCK_SIZE_SCHEDULE)} '
    f'--batch-size-schedule {shlex.quote(BATCH_SIZE_SCHEDULE)} '
    f'--optimizer {OPTIMIZER} --muon-lr {MUON_LR} --lr {LEARNING_RATE} '
    f'--lr-schedule {LR_SCHEDULE} --warmup {WARMUP_STEPS} '
    f'--out {shlex.quote(OUTPUT_DIR)} --device cuda '
    f'--resume {shlex.quote(latest)}'
)
print(cmd)
get_ipython().system(cmd)


### 6. SFT（寸法指定は不要＝checkpoint から継承）

SFT は `--init-from` の事前学習 checkpoint の `cfg` をそのまま使うので、**0.5B / 4B の寸法は自動継承**（`--preset` 不要）。渡す SentencePiece は事前学習と同一（語彙一致チェックあり）。`SFT_DATASETS` を全部連結し、assistant 発話のみ教師化する。

**効率化は事前学習と共有**（`sft_train` が `compute_loss` を再利用）: **MTP 補助損失**（checkpoint の `mtp_depth>0` を継承し、assistant マスク付きで先読みも学習）、**Muon** / **WSD** / warmup、**選択的 backprop**（`--select-topp`、既定 1.0＝無効／SFT は教師トークンが疎なので下げるなら 0.7 程度）、**μP**（`--mup`）、非有限勾配ガード。MTP は train-only なので GGUF には残らない。

**loader は spec 4 番目フィールドの 3 形式に対応**（`load_sft_conversations`）: ①会話列名（`messages`/`conversations`）、②**列マッピング DSL**（`instruction=question,reasoning=reasoning,output=answer` のように別カラムから 1 会話を組み立て、reasoning は `<think>…</think>` で包む）、③**`format=harmony`**（harmony 文字列を messages に復元、analysis チャネルは `<think>` へ）。読めない spec は警告してスキップ。3 形式は下見セル（`--sft`）でそのまま検証できる。


In [ ]:
# 6-pre. アイデンティティ（自己認識）データを生成 → SFT に --extra-jsonl で混ぜる。
#   「あなたは誰？」に「私はすずめ(suzume-dsa)です」と答えられるようにする。
#   作者句(MODEL_CREATOR)は既定で空（実在組織を騙らない）。
import shlex

cmd = (
    f'cd suzume-dsa && uv run tools/make_identity_data.py '
    f'--name {shlex.quote(MODEL_NAME)} --creator {shlex.quote(MODEL_CREATOR)} '
    f'--n {IDENTITY_N} --out identity.jsonl'
)
print(cmd)
get_ipython().system(cmd)


In [ ]:
# 6. SFT: 寸法は --init-from の checkpoint から継承（--preset 不要）。
#    効率化を全部有効化: bf16 + Muon + WSD + MTP継承 + パッキング + NEFTune + z-loss。
#    SFT_DATASETS 全部 + アイデンティティ JSONL（--extra-jsonl）を連結。
import shlex

specs = " ".join(shlex.quote(s) for s in SFT_DATASETS)
pack_flag = "--pack" if SFT_PACK else "--no-pack"
cmd = (
    f'cd suzume-dsa && uv run {os.environ.get("UV_TORCH", "")} suzume-dsa-sft '
    f'--sp-model {shlex.quote(TOKENIZER_PREFIX + ".model")} '
    f'--init-from {shlex.quote(OUTPUT_DIR + "/model.pt")} '
    f'--hf-sft-dataset {specs} '
    f'--extra-jsonl {shlex.quote(IDENTITY_JSONL)} '
    f'--steps {SFT_STEPS} --batch-size {SFT_BATCH_SIZE} --max-len {SFT_MAX_LEN} '
    f'--lr {SFT_LEARNING_RATE} --ckpt-every {SFT_CKPT_EVERY} '
    f'--optimizer {SFT_OPTIMIZER} --muon-lr {SFT_MUON_LR} '
    f'--lr-schedule {SFT_LR_SCHEDULE} --warmup {SFT_WARMUP} '
    f'--select-topp {SFT_SELECT_TOPP} --precision {PRECISION} --grad-accum {GRAD_ACCUM} '
    f'{pack_flag} --neftune {SFT_NEFTUNE} '
    f'--z-loss {SFT_Z_LOSS} --router-z {SFT_ROUTER_Z} '
    f'--out {shlex.quote(SFT_OUTPUT_DIR)} --device cuda'
)
print(cmd)
get_ipython().system(cmd)
#   → $SFT_OUTPUT_DIR/sft_model.pt（この後 ORPO で磨くか、そのまま GGUF 化）。--mup で μP も可。


### 7. ORPO（選好最適化・任意）

SFT 済みモデルを chosen/rejected の選好ペアで磨く段。**参照モデル不要**（DPO と違いメモリ 2 倍にならない＝単一GPU向き）。損失は `NLL(chosen) + λ·L_OR`（オッズ比で chosen を rejected より上げる）。応答スタイル・有用性・形式順守が上がるが、知識は増えない（そこは規模×データ）。SFT の寸法・語彙をそのまま継承する。スキップして SFT 済みをそのまま GGUF 化してもよい。


In [ ]:
# 7. ORPO: SFT 済み(sft_model.pt)を選好データで磨く。寸法は checkpoint から継承。
import shlex

specs = " ".join(shlex.quote(s) for s in PREF_DATASETS)
cmd = (
    f'cd suzume-dsa && uv run {os.environ.get("UV_TORCH", "")} suzume-dsa-orpo '
    f'--sp-model {shlex.quote(TOKENIZER_PREFIX + ".model")} '
    f'--init-from {shlex.quote(SFT_OUTPUT_DIR + "/sft_model.pt")} '
    f'--hf-pref-dataset {specs} '
    f'--steps {ORPO_STEPS} --batch-size {ORPO_BATCH_SIZE} --max-len {ORPO_MAX_LEN} '
    f'--lr {ORPO_LR} --lam {ORPO_LAM} --precision {PRECISION} --grad-accum {GRAD_ACCUM} '
    f'--out {shlex.quote(ORPO_OUTPUT_DIR)} --device cuda'
)
print(cmd)
get_ipython().system(cmd)
#   → $ORPO_OUTPUT_DIR/orpo_model.pt（次の GGUF 化はこれ。ORPO を飛ばすなら sft_model.pt）


### 8. GGUF エクスポート（名乗り付き）

最終チェックポイント（ORPO があれば `orpo_model.pt`、無ければ SFT の `sft_model.pt`）を GGUF 化する。`--name` で `general.name`、`--default-system` で ChatML の `chat_template`（既定システムプロンプトに名乗りを注入）を埋め込むので、llama.cpp で読むと自己識別・正しいチャット整形が効く。cfg は checkpoint から、語彙は SP から自動。


In [ ]:
# 8. GGUF エクスポート。ORPO 済みがあればそれを、無ければ SFT 済みを使う。
import os
import shlex

CKPT = (ORPO_OUTPUT_DIR + "/orpo_model.pt"
        if os.path.exists(ORPO_OUTPUT_DIR + "/orpo_model.pt")
        else SFT_OUTPUT_DIR + "/sft_model.pt")
print("export:", CKPT)
cmd = (
    f'cd suzume-dsa && uv run suzume-dsa-export '
    f'--checkpoint {shlex.quote(CKPT)} --sp-model {shlex.quote(TOKENIZER_PREFIX + ".model")} '
    f'--name {shlex.quote(MODEL_NAME)} --default-system {shlex.quote(DEFAULT_SYSTEM)} '
    f'--out {shlex.quote(GGUF_OUT)}'
)
print(cmd)
get_ipython().system(cmd)

# 量子化して実行（llama.cpp をビルド済みの場合）:
# !/workspace/llama.cpp/build/bin/llama-quantize {GGUF_OUT} {GGUF_OUT.replace(".gguf","-Q4_K_M.gguf")} Q4_K_M
# !/workspace/llama.cpp/build/bin/llama-cli -m {GGUF_OUT.replace(".gguf","-Q4_K_M.gguf")} -p "あなたは誰？" -n 128
